In [ ]:
!pip install psycopg2-binary -q

In [ ]:
## List of available filter years (view)

view_years = '''
CREATE OR REPLACE VIEW public.vw_years_since_2020 AS
SELECT generate_series(
    2020,
    CASE 
        WHEN CURRENT_DATE > date_trunc('year', CURRENT_DATE) + interval '1 month 23 days'
        THEN EXTRACT(YEAR FROM CURRENT_DATE)::int
        ELSE EXTRACT(YEAR FROM CURRENT_DATE)::int - 1
    END
) AS annee;
'''

# Create view
from sqlalchemy import create_engine, text
from openhexa.sdk import workspace
engine = create_engine(workspace.database_url)
with engine.begin() as conn:
    conn.execute(text(view_years))


In [ ]:
## Provinces PNLP

view_prov_pnlp = '''
CREATE OR REPLACE VIEW public.vw_pnlp_palu_tot_prov_geometry AS
SELECT 
    cases_table.*,
    shapes.first_polygon
FROM (
    SELECT 
        "year",        
        "PROVINCE",
        SUM("Cas")   AS "Cas",
        SUM("Deces") AS "Deces"
    FROM public."DSE_palu_for_PNLP_tot"
    GROUP BY "year", "PROVINCE"
) AS cases_table
LEFT JOIN public."cod_iaso_provinces_ss" AS shapes
    ON cases_table."PROVINCE" = shapes.province;
'''

# Create view
from sqlalchemy import create_engine, text
from openhexa.sdk import workspace
engine = create_engine(workspace.database_url)
with engine.begin() as conn:
    conn.execute(text(view_prov_pnlp))

# Usage
# select * from public.vw_pnlp_palu_tot_prov_geometry where year = 2024

In [ ]:
## Zones de sante PNLP

view_zs_pnlp='''
CREATE OR REPLACE VIEW public.vw_pnlp_palu_tot_zs_geometry AS
SELECT
    g.zone_de_sante,
    g.first_polygon,
    d."year",
    d."PROVINCE",
    d."Nom",
    COALESCE(d.total_cas, 0)   AS total_cas,
    COALESCE(d.total_deces, 0) AS total_deces
FROM public."cod_iaso_zone_de_sante_ss" g
LEFT JOIN (
    SELECT  
        "Nom",
        "PROVINCE",
        "year",
        SUM("Cas")   AS total_cas,
        SUM("Deces") AS total_deces
    FROM public."DSE_palu_for_PNLP_tot"	
    GROUP BY "Nom", "PROVINCE", "year"
) d
ON g.zone_de_sante = d."Nom"
ORDER BY g.zone_de_sante, d."year";
'''

# Create view
from sqlalchemy import create_engine, text
from openhexa.sdk import workspace
engine = create_engine(workspace.database_url)
with engine.begin() as conn:
    conn.execute(text(view_zs_pnlp))
    
# Usage
# select count(*) from public."vw_pnlp_palu_tot_zs_geometry" where year = 2023

In [ ]:
## Indicator aggregations (proportion category - provinces) 

view_ind_prov = '''
CREATE OR REPLACE VIEW public.vw_indicators_proportion_prov as
 SELECT indicators.*, shapes.first_polygon
   FROM "indicators_ND_agg_proportion" indicators
     LEFT JOIN cod_iaso_provinces_ss shapes ON indicators.province = shapes.province
WHERE indicators.aggregation_spatiale = 'Province' 
      AND indicators.aggregation_temporelle = 'Mois';
'''

# Create view
from sqlalchemy import create_engine, text
from openhexa.sdk import workspace
engine = create_engine(workspace.database_url)
with engine.begin() as conn:
    conn.execute(text(view_ind_prov))


In [ ]:
## Indicator aggregations (proportion category - ZS)

view_ind_zs = '''
CREATE OR REPLACE VIEW public.vw_indicators_proportion_zs as
SELECT indicators.*, shapes.first_polygon
    FROM "indicators_ND_agg_proportion" indicators
       LEFT JOIN cod_iaso_zone_de_sante_ss shapes ON indicators.zone_de_sante = shapes.zone_de_sante
WHERE indicators.aggregation_spatiale = 'Zone' 
      AND indicators.aggregation_temporelle = 'Mois'
'''

# Create view
from sqlalchemy import create_engine, text
from openhexa.sdk import workspace
engine = create_engine(workspace.database_url)
with engine.begin() as conn:
    conn.execute(text(view_ind_zs))
